# Datenbasis für die EQL-Demo

Dieses Notebook erzeugt die In-Memory-SQLite-Datenbank und die Beispielobjekte,
die `eql_demo.ipynb` und `eql_test.ipynb` benutzen. Beide binden dieses
Notebook per `%run ./eql_data_setup.ipynb` ein — Datenaufbau ist damit an
genau einer Stelle definiert, nicht in jedem Notebook einzeln.

**Wichtiger technischer Hinweis:** `krrood_test/dataset/ormatic_interface.py`
(mit `Base`, `BodyDAO`, `ContainerDAO`, ...) liegt nicht als statische Datei
im Repo, sondern wird beim Import von `krrood_test.conftest` als Nebeneffekt
generiert (`generate_sqlalchemy_interface()` läuft dort unconditional auf
Modulebene, während pytest die Conftest-Datei sammelt). Ein direkter Import
von `krrood_test.conftest` im Notebook-Kernel selbst ist fragil — dabei
fehlten zunächst Sys-Path-Einträge und anschließend traten sogar doppelt
generierte Tabellen auf, weil dieser Weg nicht denselben Import-Ablauf
durchläuft wie ein echter pytest-Lauf.

Der robustere Weg (und das, was hier jetzt passiert): ein echter, minimaler
`pytest --collect-only`-Lauf wird per `subprocess` in einem eigenen Prozess
gestartet — das reicht, weil `ormatic_interface.py` schon beim reinen
Einsammeln der Tests entsteht, nicht erst beim Ausführen. Pytest kümmert
sich dabei selbst um Pfade und Importreihenfolge, genau wie bei jedem
normalen Testlauf. Erst danach importiert der Notebook-Kernel die fertig
generierte Datei ganz normal — ohne `krrood_test.conftest` je selbst
anzufassen.

## 1. Session

In [1]:
import subprocess
import sys

TEST_ROOT = "/home/jovyan/krrood/test"  # Pfad ggf. anpassen, siehe Dockerfile

# Beide Pfade werden für den Notebook-Kernel selbst gebraucht: manche Module
# in krrood_test importieren sich absolut als "test.krrood_test...."
for _p in (TEST_ROOT, "/home/jovyan/krrood"):
    if _p not in sys.path:
        sys.path.insert(0, _p)

# Erzwingt die (Re-)Generierung von dataset/ormatic_interface.py über einen
# echten pytest-Lauf statt eines direkten Imports von krrood_test.conftest im
# Notebook-Kernel (siehe Hinweis oben). --collect-only reicht, weil die
# Generierung schon beim Einsammeln passiert -- kein Testlauf nötig, deutlich
# schneller als der volle Lauf.
_result = subprocess.run(
    [sys.executable, "-m", "pytest", "krrood_test/test_ormatic/test_eql.py",
     "-q", "--collect-only"],
    cwd=TEST_ROOT, capture_output=True, text=True,
)
if _result.returncode != 0:
    print(_result.stdout[-3000:])
    print(_result.stderr[-3000:])
    raise RuntimeError(
        "Generierung von ormatic_interface.py fehlgeschlagen (siehe Ausgabe oben)."
    )
print("ormatic_interface.py generiert (über einen echten pytest --collect-only-Lauf).")

from sqlalchemy.orm import sessionmaker, configure_mappers

from krrood.ormatic.utils import create_engine
from test.krrood_test.dataset.ormatic_interface import Base, BodyDAO
from test.krrood_test.dataset.semantic_world_like_classes import (
    Container,
    Handle,
    FixedConnection,
    PrismaticConnection,
    World,
)


def create_demo_session():
    """
    Frische In-Memory-SQLite-Session, aufgebaut genau wie die echten
    engine/session/database-Fixtures in krrood_test/conftest.py.

    Jeder Aufruf legt eine neue, leere Datenbank an -- wer diese Funktion
    erneut aufruft, fängt bei einer sauberen Datenbasis an, statt Reste aus
    vorherigen Zellen mitzuschleppen.
    """
    configure_mappers()
    engine = create_engine("sqlite:///:memory:")
    Base.metadata.create_all(engine)
    Session = sessionmaker(bind=engine)
    return Session()


print("create_demo_session() definiert.")

ormatic_interface.py generiert (über einen echten pytest --collect-only-Lauf).
create_demo_session() definiert.


## 2. Bodies unterschiedlicher Größe (für Simple Filter / Complex Combined)

Für `b.size > 5` braucht es Zeilen auf beiden Seiten des Schwellwerts, sonst
filtert die WHERE-Klausel sichtbar nichts heraus und das Beispiel wirkt
beliebig.

In [2]:
def build_bodies_for_simple_filter(session, sizes=(1, 3, 6, 8, 10)):
    """
    Legt Body-Zeilen mit den angegebenen Größen an.

    :param session: Aktive SQLAlchemy-Session (siehe create_demo_session)
    :param sizes: Größen der angelegten Bodies
    :return: Anzahl der eingefügten Zeilen
    """
    for i, size in enumerate(sizes, start=1):
        session.add(BodyDAO(name=f"Body{i}", size=size))
    session.commit()
    return len(sizes)


print("build_bodies_for_simple_filter() definiert.")

build_bodies_for_simple_filter() definiert.


## 3. Container/Handle/Connection-Welten (für die Multi-Join-Fälle)

Diese Objekte sind Domain-Objekte aus `semantic_world_like_classes`, die EQL
für Typauflösung und Domain-Scoping braucht (`variable(Container,
domain=world.bodies)`). Für eine echte `.evaluate()`-Ausführung mit
Ergebniszeilen müssten zusätzlich passende `ContainerDAO`/`HandleDAO`/
`FixedConnectionDAO`-Zeilen in der Session persistiert werden — das ist hier
bewusst nicht nachgebaut, siehe Hinweis in `eql_demo.ipynb`.

In [3]:
def build_container_handle_world():
    """
    Container -- FixedConnection -- Handle: das 3-Variablen-Beispiel
    (C, H, FC) aus Kapitel 6.

    :return: World-Instanz mit einem Container, einem Handle und einer
        verbindenden FixedConnection
    """
    world = World(1, [Container('C1'), Handle('H1')])
    fc = FixedConnection(world.bodies[0], world.bodies[1])
    world.connections = [fc]
    return world


def build_four_variable_world():
    """
    Zwei Container, zwei Handles, eine FixedConnection und eine
    PrismaticConnection — das Skalierungsbeispiel mit 4 Variablen
    (C, H, FC, PC) aus Kapitel 6.

    :return: World-Instanz mit den vier verbundenen Körpern
    """
    world = World(1, [
        Container('C1'), Container('C2'),
        Handle('H1'), Handle('H2'),
    ])
    fc = FixedConnection(world.bodies[0], world.bodies[2])
    pc = PrismaticConnection(world.bodies[1], world.bodies[0])
    world.connections = [fc, pc]
    return world


def build_container_chain(k: int):
    """
    Kette aus k Containern, verbunden über (k - 1) FixedConnections:

        C0 --FC0-- C1 --FC1-- C2 --...-- C(k-1)

    Für die Skalierungsbetrachtung über mehrere Variablenzahlen (nicht nur 3
    und 4), ohne dafür neue Domain-Klassen zu brauchen.

    :param k: Anzahl der Container in der Kette (k >= 2)
    :return: World-Instanz mit der Kette
    """
    bodies = [Container(f'C{i}') for i in range(k)]
    world = World(1, bodies)
    connections = [FixedConnection(bodies[i], bodies[i + 1]) for i in range(k - 1)]
    world.connections = connections
    return world


print("build_container_handle_world(), build_four_variable_world(), build_container_chain() definiert.")

build_container_handle_world(), build_four_variable_world(), build_container_chain() definiert.


## 4. Kurzer Funktionstest

Nur zur Kontrolle, dass alles oben tatsächlich funktioniert — erzeugt keine
Variablen, die die anderen Notebooks überschreiben würden (die rufen die
Funktionen selbst mit ihren eigenen Variablennamen auf).

In [4]:
_check_session = create_demo_session()
_n = build_bodies_for_simple_filter(_check_session)
print(f"OK: {_n} Bodies angelegt.")

_check_world_3 = build_container_handle_world()
print(f"OK: 3-Variablen-Welt mit {len(_check_world_3.bodies)} Bodies, {len(_check_world_3.connections)} Connection.")

_check_world_4 = build_four_variable_world()
print(f"OK: 4-Variablen-Welt mit {len(_check_world_4.bodies)} Bodies, {len(_check_world_4.connections)} Connections.")

_check_chain = build_container_chain(4)
print(f"OK: Kette mit {len(_check_chain.bodies)} Containern, {len(_check_chain.connections)} Connections.")

del _check_session, _n, _check_world_3, _check_world_4, _check_chain

OK: 5 Bodies angelegt.
OK: 3-Variablen-Welt mit 2 Bodies, 1 Connection.
OK: 4-Variablen-Welt mit 4 Bodies, 2 Connections.
OK: Kette mit 4 Containern, 3 Connections.
